In [0]:
import requests

import pandas as pd

from pyspark.sql import SparkSession

from pyspark.sql import functions as F

from delta.tables import DeltaTable

spark = SparkSession.builder.getOrCreate()

# =========================

# 🔐 GET SECRETS (FIXED)

# =========================

BASE_URL = dbutils.secrets.get(scope="slainte-secrets", key="supabase-url").strip()

API_KEY  = dbutils.secrets.get(scope="slainte-secrets", key="supabase-api-key").strip()

HEADERS = {

    "apikey": API_KEY,

    "Authorization": f"Bearer {API_KEY}"

}

# =========================

# 📡 FETCH DATA

# =========================

resp = requests.get(f"{BASE_URL}/jira_integrations", headers=HEADERS)

print("Status:", resp.status_code)

# ❌ Stop early if API fails

if resp.status_code != 200:

    print("❌ API ERROR:", resp.text[:300])

    raise Exception("API request failed")

# =========================

# 🧱 CREATE PANDAS DF

# =========================

pdf = pd.DataFrame(resp.json()).where(lambda x: x.notna(), None)

# Safety check

if pdf.empty:

    print("⚠️ No data returned from API")

    raise Exception("Empty dataset")

# =========================

# 🧹 SELECT COLUMNS

# =========================

pdf = pdf[[

    "id", "user_id", "project_name", "site_url", "user_email",

    "api_token", "status", "connection_tested",

    "created_at", "updated_at", "step", "project_key"

]]

# =========================

# ⚡ CONVERT TO SPARK

# =========================

df = spark.createDataFrame(pdf).withColumn(

    "ingestion_timestamp", F.current_timestamp()

)

# =========================

# 🧠 DELTA MERGE

# =========================

TARGET = "workspace.slainte_bronze.jira_integrations_bronze"

if not spark.catalog.tableExists(TARGET):

    df.write.mode("overwrite").format("delta").saveAsTable(TARGET)

    print("✅ Table created (initial load)")

else:

    delta_table = DeltaTable.forName(spark, TARGET)

    (

        delta_table.alias("t")

        .merge(

            df.alias("s"),

            "t.id = s.id"

        )

        .whenMatchedUpdate(set={

            "user_id": "s.user_id",

            "project_name": "s.project_name",

            "site_url": "s.site_url",

            "user_email": "s.user_email",

            "api_token": "s.api_token",

            "status": "s.status",

            "connection_tested": "s.connection_tested",

            "created_at": "s.created_at",

            "updated_at": "s.updated_at",

            "step": "s.step",

            "project_key": "s.project_key",

            "ingestion_timestamp": "s.ingestion_timestamp"

        })

        .whenNotMatchedInsertAll()

        .execute()

    )

    print("✅ Table merged (incremental)")

# =========================

# 📊 FINAL CHECK

# =========================

print("Rows:", df.count())
 